# 05 ― プロジェクトのパレットで独自チャート

アプリ本体の天気図 (t2m, MSL, TP …) は `figures/_palette.py` の
**256 エントリ連続 LUT** で色付けしている。同じ LUT を notebook 側でも
import すれば、自前の解析結果を **アプリと完全に揃った色で**表示できる。

ここでは t2m (2 m 気温) のアンカーを使って、ダミーの温度フィールドを
プロットしてみる。実データ ― ECMWF GRIB2 や Open-Meteo の DataFrame ―
に同じパターンを適用すれば、出版品質の図がそのまま手に入る。

**読み合わせ**: `.agents/skills/chart-base-design/SKILL.md` (パレット規約)


## パレット LUT を構築


In [ ]:
from aiseed_weather.figures._palette import build_continuous_lut
from aiseed_weather.figures._chart_specs import T2M

# プロジェクト本体の t2m チャートと完全に同じアンカー / vmin / vmax
print(f"variable    : {T2M.label}")
print(f"value range : {T2M.vmin}°C 〜 {T2M.vmax}°C")
print(f"anchors     : {len(T2M.anchors)} 色")
for v, rgb in T2M.anchors:
    print(f"  {v:+6.1f}°C  →  RGB{rgb}")

lut = build_continuous_lut(T2M.anchors, T2M.vmin, T2M.vmax)
print(f"\nLUT shape   : {lut.shape}  dtype={lut.dtype}")


## カラーバーで LUT を視覚化


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

# matplotlib 用に正規化 (uint8 → float [0, 1])
cmap = ListedColormap(lut / 255.0, name="aiseed_t2m")
norm = plt.Normalize(vmin=T2M.vmin, vmax=T2M.vmax)

fig, ax = plt.subplots(figsize=(8, 1.2))
sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(sm, cax=ax, orientation="horizontal")
cbar.set_ticks(list(T2M.legend_ticks))
cbar.set_label("気温 / Temperature (°C)")
plt.show()


## ダミーの温度場を描く

緯度依存 (赤道で暖、極で寒) + 経度方向の波で「それっぽい」フィールドを生成。


In [ ]:
lats = np.linspace(50, 20, 121)  # 北→南
lons = np.linspace(120, 150, 121)
LAT, LON = np.meshgrid(lats, lons, indexing="ij")

# 気候学的に妥当な範囲のダミー: 緯度 35° で 15°C 基準、±5°C/緯度10°、波
field = 15.0 - 0.5 * (LAT - 35.0) + 4.0 * np.sin(np.deg2rad(LON * 4))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(LON, LAT, field, cmap=cmap, norm=norm, shading="auto")
ax.set_xlabel("経度 / Longitude (°E)")
ax.set_ylabel("緯度 / Latitude (°N)")
ax.set_title("ダミー温度場 ― aiseed_t2m パレット")

cbar = fig.colorbar(im, ax=ax, ticks=list(T2M.legend_ticks))
cbar.set_label("気温 (°C)")

# === 帰属表示 ===
# 本物のデータを描くときは、出典 / ライセンス / 取得時刻を必ず入れる。
# プロジェクト本体は figures/footer.py を共通フッタとして使っている。
fig.text(
    0.99, 0.01,
    "Demo data only. Production figures: include source, run time, license.",
    ha="right", va="bottom", fontsize=7, color="#888",
)
plt.show()


## 他のパレット (降水・気圧) も同様に使える


In [ ]:
from aiseed_weather.figures._chart_specs import TP  # 積算降水量

tp_lut = build_continuous_lut(TP.anchors, TP.vmin, TP.vmax)
tp_cmap = ListedColormap(tp_lut / 255.0, name="aiseed_tp")

fig, ax = plt.subplots(figsize=(8, 1.2))
sm = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=TP.vmin, vmax=TP.vmax), cmap=tp_cmap,
)
cbar = fig.colorbar(sm, cax=ax, orientation="horizontal")
cbar.set_ticks(list(TP.legend_ticks))
cbar.set_label("積算降水量 / Precipitation (mm)")
plt.show()


## ワークフロー: 実データに同じパレットを当てる

1. `02-ecmwf-grib2.ipynb` で xarray Dataset を取得
2. 同じ extractor (`T2M.extractor(ds)`) で 2D 配列に変換
3. `build_continuous_lut(T2M.anchors, T2M.vmin, T2M.vmax)` で LUT 構築
4. `ax.pcolormesh(..., cmap=ListedColormap(lut/255), norm=Normalize(vmin, vmax))`
5. `figures/footer.py` の `apply_footer(fig, source=..., run_time=...)` で帰属表示

これでアプリ本体と完全に同じ見た目の図がノートブック側でも作れる。
新しい変数を試作したい場合 ― 例えば「等温位面の風速」 ― 既存のアンカー
を流用して試し、最終的にアプリ本体の `_chart_specs.py` に
`register(ChartSpec(...))` で追加する、という流れがプロジェクトの定石。
